In [1]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

CACHE_DIR = Path("cache")
ANN = 252

CACHE_FILES = {
    "ML_Conservative":      "ML_Conservative_a41982e916.pkl",
    "ML_Moderate":          "ML_Moderate_f57aa66187.pkl",
    "ML_Aggressive":        "ML_Aggressive_e74f88ad7e.pkl",

    "Jump_Conservative":    "Jump_Conservative_32d4a9e7fa.pkl",
    "Jump_Moderate":        "Jump_Moderate_b6b1092064.pkl",
    "Jump_Aggressive":      "Jump_Aggressive_c4edbb0158.pkl",

    "HMM_Conservative":     "HMM_Conservative_76116f29a4.pkl",
    "HMM_Moderate":         "HMM_Moderate_6d4b017b3b.pkl",
    "HMM_Aggressive":       "HMM_Aggressive_8cef21daf0.pkl",

    "HSMM_Conservative":    "HSMM_Conservative_855bed7ed1.pkl",
    "HSMM_Moderate":        "HSMM_Moderate_4d440ea00f.pkl",
    "HSMM_Aggressive":      "HSMM_Aggressive_1b376820dc.pkl",
}

def compute_metrics(bt_wf, bt_bm):
    r   = bt_wf["ret"]
    cum = bt_wf["cum"]
    bm_r = bt_bm["ret"].reindex(r.index).fillna(0)

    ar   = (1+r).prod()**(ANN/len(r)) - 1
    vol  = r.std() * np.sqrt(ANN)
    sh   = r.mean()*ANN / vol if vol > 0 else 0
    neg  = r[r < 0]
    dv   = neg.std() * np.sqrt(ANN) if len(neg) > 0 else np.nan
    so   = r.mean()*ANN / dv if dv and dv > 0 else 0
    dd   = ((cum - cum.cummax()) / cum.cummax()).min()
    cal  = ar / abs(dd) if dd != 0 else 0
    var  = np.percentile(r, 5)
    cvar = r[r <= var].mean()
    sk   = r.skew()
    ku   = r.kurt()  # excess kurtosis
    sw   = int(bt_wf.get("rebalanced", pd.Series(dtype=bool)).sum())
    wr   = round(((1+r).resample("ME").prod()-1 > 0).mean()*100, 1)

    bm_ar  = (1+bm_r).prod()**(ANN/len(bm_r)) - 1
    bm_vol = bm_r.std() * np.sqrt(ANN)
    bm_sh  = bm_r.mean()*ANN / bm_vol if bm_vol > 0 else 0
    bm_dd  = ((bt_bm["cum"] - bt_bm["cum"].cummax()) / bt_bm["cum"].cummax()).min()

    return {
        "Ann. Return":       f"{ar*100:.2f}%",
        "Ann. Vol":          f"{vol*100:.2f}%",
        "Sharpe":            f"{sh:.3f}",
        "Sortino":           f"{so:.3f}",
        "Calmar":            f"{cal:.3f}",
        "Max Drawdown":      f"{dd*100:.1f}%",
        "VaR (95%)":         f"{var*100:.3f}%",
        "CVaR (95%)":        f"{cvar*100:.3f}%",
        "Skewness":          f"{sk:.3f}",
        "Excess Kurtosis":   f"{ku:.3f}",
        "Win Rate":          f"{wr}%",
        "Switches":          str(sw),
        "Sharpe Alpha":      f"{sh-bm_sh:.3f}",
        "BM Ann. Return":    f"{bm_ar*100:.2f}%",
        "BM Sharpe":         f"{bm_sh:.3f}",
        "BM Max Drawdown":   f"{bm_dd*100:.1f}%",
    }

rows = []
for label, fname in CACHE_FILES.items():
    path = CACHE_DIR / fname
    if not path.exists():
        print(f"Missing: {fname}")
        continue
    with open(path, "rb") as f:
        cache = pickle.load(f)
    metrics = compute_metrics(cache["bt_wf"], cache["bt_bm"])
    metrics["Model_Profile"] = label
    rows.append(metrics)

df = pd.DataFrame(rows).set_index("Model_Profile")
df = df[["Ann. Return","Ann. Vol","Sharpe","Sortino","Calmar",
         "Max Drawdown","VaR (95%)","CVaR (95%)","Skewness",
         "Excess Kurtosis","Win Rate","Switches","Sharpe Alpha",
         "BM Ann. Return","BM Sharpe","BM Max Drawdown"]]

print(df.to_string())
df.to_csv("appendix_e_risk_metrics.csv")
print("\nSaved to appendix_e_risk_metrics.csv")

                  Ann. Return Ann. Vol Sharpe Sortino Calmar Max Drawdown VaR (95%) CVaR (95%) Skewness Excess Kurtosis Win Rate Switches Sharpe Alpha BM Ann. Return BM Sharpe BM Max Drawdown
Model_Profile                                                                                                                                                                                  
ML_Conservative        11.12%   10.13%  1.092   1.518  0.499       -22.3%   -1.007%    -1.480%   -0.226           2.546    66.7%       34        0.432          5.61%     0.659          -22.0%
ML_Moderate            13.40%   12.35%  1.081   1.463  0.541       -24.8%   -1.254%    -1.851%   -0.295           2.946    66.7%       34        0.371          8.38%     0.710          -22.4%
ML_Aggressive          13.98%   14.90%  0.953   1.248  0.597       -23.4%   -1.509%    -2.238%   -0.376           5.514    67.9%       34        0.151         13.26%     0.802          -30.7%
Jump_Conservative      11.59%   11.69%  

In [11]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

CACHE_DIR = Path("cache")
ANN = 252


CACHE_FILES = {
    "Conservative": {
        "ML":   "ML_Conservative_a41982e916.pkl",
        "Jump": "Jump_Conservative_32d4a9e7fa.pkl",
        "HMM":  "HMM_Conservative_76116f29a4.pkl",
        "HSMM": "HSMM_Conservative_855bed7ed1.pkl",
    },
    "Moderate": {
        "ML":   "ML_Moderate_f57aa66187.pkl",
        "Jump": "Jump_Moderate_b6b1092064.pkl",
        "HMM":  "HMM_Moderate_6d4b017b3b.pkl",
        "HSMM": "HSMM_Moderate_4d440ea00f.pkl",
    },
    "Aggressive": {
        "ML":   "ML_Aggressive_e74f88ad7e.pkl",
        "Jump": "Jump_Aggressive_c4edbb0158.pkl",
        "HMM":  "HMM_Aggressive_8cef21daf0.pkl",
        "HSMM": "HSMM_Aggressive_1b376820dc.pkl",
    },
}

METRICS_ORDER = [
    "Ann. Return", "Ann. Vol", "Sharpe", "Sortino", "Calmar",
    "Max Drawdown", "VaR (95%)", "CVaR (95%)", "Skewness",
    "Excess Kurtosis", "Win Rate", "Switches", "Sharpe Alpha",
]

def compute_metrics(bt_wf, bt_bm):
    r    = bt_wf["ret"]
    cum  = bt_wf["cum"]
    bm_r = bt_bm["ret"].reindex(r.index).fillna(0)

    ar   = (1+r).prod()**(ANN/len(r)) - 1
    vol  = r.std() * np.sqrt(ANN)
    sh   = r.mean()*ANN / vol if vol > 0 else 0
    neg  = r[r < 0]
    dv   = neg.std() * np.sqrt(ANN) if len(neg) > 0 else np.nan
    so   = r.mean()*ANN / dv if dv and dv > 0 else 0
    dd   = ((cum - cum.cummax()) / cum.cummax()).min()
    cal  = ar / abs(dd) if dd != 0 else 0
    var  = np.percentile(r, 5)
    cvar = r[r <= var].mean()
    sk   = r.skew()
    ku   = r.kurt()
    sw   = int(bt_wf.get("rebalanced", pd.Series(dtype=bool)).sum())
    wr   = round(((1+r).resample("ME").prod()-1 > 0).mean()*100, 1)
    bm_vol = bm_r.std() * np.sqrt(ANN)
    bm_sh  = bm_r.mean()*ANN / bm_vol if bm_vol > 0 else 0

    return {
        "Ann. Return":     f"{ar*100:.2f}%",
        "Ann. Vol":        f"{vol*100:.2f}%",
        "Sharpe":          f"{sh:.3f}",
        "Sortino":         f"{so:.3f}",
        "Calmar":          f"{cal:.3f}",
        "Max Drawdown":    f"{dd*100:.1f}%",
        "VaR (95%)":       f"{var*100:.3f}%",
        "CVaR (95%)":      f"{cvar*100:.3f}%",
        "Skewness":        f"{sk:.3f}",
        "Excess Kurtosis": f"{ku:.3f}",
        "Win Rate":        f"{wr}%",
        "Switches":        str(sw),
        "Sharpe Alpha":    f"{sh-bm_sh:.3f}",
    }

def compute_bm_metrics(bt_bm):
    r   = bt_bm["ret"]
    cum = bt_bm["cum"]
    ar  = (1+r).prod()**(ANN/len(r)) - 1
    vol = r.std() * np.sqrt(ANN)
    sh  = r.mean()*ANN / vol if vol > 0 else 0
    neg = r[r < 0]
    dv  = neg.std() * np.sqrt(ANN) if len(neg) > 0 else np.nan
    so  = r.mean()*ANN / dv if dv and dv > 0 else 0
    dd  = ((cum - cum.cummax()) / cum.cummax()).min()
    cal = ar / abs(dd) if dd != 0 else 0
    var = np.percentile(r, 5)
    cvar= r[r <= var].mean()
    sk  = r.skew()
    ku  = r.kurt()
    wr  = round(((1+r).resample("ME").prod()-1 > 0).mean()*100, 1)

    return {
        "Ann. Return":     f"{ar*100:.2f}%",
        "Ann. Vol":        f"{vol*100:.2f}%",
        "Sharpe":          f"{sh:.3f}",
        "Sortino":         f"{so:.3f}",
        "Calmar":          f"{cal:.3f}",
        "Max Drawdown":    f"{dd*100:.1f}%",
        "VaR (95%)":       f"{var*100:.3f}%",
        "CVaR (95%)":      f"{cvar*100:.3f}%",
        "Skewness":        f"{sk:.3f}",
        "Excess Kurtosis": f"{ku:.3f}",
        "Win Rate":        f"{wr}%",
        "Switches":        "0",
        "Sharpe Alpha":    "0.000",
    }

for profile, model_files in CACHE_FILES.items():
    print(f"\n{'='*70}")
    print(f"RISK PROFILE: {profile}")
    print(f"{'='*70}")

    cols = {}
    bm_added = False

    # CORRECT
    for model, fname in model_files.items():
        path = CACHE_DIR / fname
        if not path.exists():
            print(f"  Missing: {fname}")
            continue
        with open(path, "rb") as f:
            cache = pickle.load(f)

        # Add benchmark once per profile
        if not bm_added:
            cols["Static BM"] = compute_bm_metrics(cache["bt_bm"])
            bm_added = True

        cols[model] = compute_metrics(cache["bt_wf"], cache["bt_bm"])

    if cols:
        df = pd.DataFrame(cols).loc[METRICS_ORDER]
        print(df.to_string())
        df.to_csv(f"appendix_e_{profile}.csv")
        print(f"\nSaved: appendix_e_{profile}.csv")


RISK PROFILE: Conservative
                Static BM       ML     Jump      HMM     HSMM
Ann. Return         5.61%   11.12%   11.59%    9.15%    7.46%
Ann. Vol            8.87%   10.13%   11.69%   10.82%   12.30%
Sharpe              0.659    1.092    0.997    0.864    0.647
Sortino             0.848    1.518    1.391    1.131    0.871
Calmar              0.255    0.499    0.400    0.326    0.258
Max Drawdown       -22.0%   -22.3%   -29.0%   -28.0%   -29.0%
VaR (95%)         -0.847%  -1.007%  -1.138%  -1.101%  -1.217%
CVaR (95%)        -1.308%  -1.480%  -1.700%  -1.650%  -1.807%
Skewness           -0.292   -0.226    0.074   -0.272   -0.089
Excess Kurtosis     8.964    2.546    4.961    3.799    7.057
Win Rate            63.1%    66.7%    67.9%    66.7%    59.5%
Switches                0       34        8       31       25
Sharpe Alpha        0.000    0.432    0.338    0.204   -0.013

Saved: appendix_e_Conservative.csv

RISK PROFILE: Moderate
                Static BM       ML     Jump 

In [9]:
model_files

'ML_Conservative_a41982e916.pkl'